In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif, SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression

from sklearn.datasets import load_iris

In [ ]:
# Q1
# Chi-Square Test for Feature Selection
# Use the Iris dataset and implement feature selection using the Chi-square test

iris = load_iris()
X = iris.data
y= iris.target
feature_names = iris.feature_names

# Standardize data to [0,1] as chi-square testing assume non-negative values
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

selector = SelectKBest(score_func=chi2, k='all')
selector.fit(X_scaled, y)
scores = selector.scores_

results = pd.DataFrame({
    'feature': feature_names,
    'chi2_score': scores,
}).sort_values(by='chi2_score', ascending=False).reset_index(drop=True)

print('\nRanked features based on chi2 score:')
for i, row in results.iterrows():
    print(f"{i+1}. {row['feature']}  chi2={row['chi2_score']:.5f}")

In [ ]:
# Q2
# Information Gain for Feature Selection
# Use the Iris dataset and implement feature selection using Information Gain (Mutual Information)

iris = load_iris()
X = iris.data
y= iris.target
feature_names = iris.feature_names

mutual_info_scores = mutual_info_classif(X, y)

results = pd.DataFrame({
    'feature': feature_names,
    'mutual information': mutual_info_scores
}).sort_values(by='mutual information', ascending=False).reset_index(drop=True)

print('\nRanked features based on mutual information score:')
for i, row in results.iterrows():
    print(f"{i+1}. {row['feature']}  mutual_info={row['mutual information']:.5f}")

In [ ]:
# Q3 
# Sequential Forward Selection (SFS)
# Implement SFS for feature selection using the Iris dataset. Track accuracy of a k-NN classifier as features are added incrementally

iris = load_iris()
X = iris.data
y= iris.target
feature_names = np.array(iris.feature_names)

# Pipeline for scaling
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# Cross validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Track accuracy as features are added
results = []
for k in range(1, X.shape[1]):
    sfs = SequentialFeatureSelector(
        estimator=knn_pipe,
        n_features_to_select=k,
        direction='forward',
        scoring='accuracy',
        cv=cv,
        n_jobs=-1
    )
    sfs.fit(X, y)

    selected_mask = sfs.get_support()
    selected_features = feature_names[selected_mask].tolist()

    # Accuracy using only the selected features
    acc = cross_val_score(knn_pipe, X[:, selected_mask], y, cv=cv, scoring='accuracy').mean()

    results.append({
        'num_features': k,
        'selected_features': selected_features,
        'cv_accuracy': acc
    })

# Accuracy with all 4 features
acc = cross_val_score(knn_pipe, X, y, cv=cv, scoring='accuracy').mean()
results.append({
    'num_features': 4,
    'selected_features': feature_names,
    'cv_accuracy': acc
})

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

In [ ]:
# Q4
# Random Forest for Feature Importance
# Use the Random Forest Classifier from sklearn to compute feature importance on the iris dataset. 
# Plot the importance of scores of each feature and evaluate how feature importance correlates with classification accuracy.
# Explain the differences and reasons of SFS and Radom Forest
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names

rfc = RandomForestClassifier(n_estimators=200, random_state=42)
rfc.fit(X, y)
feature_importance = rfc.feature_importances_

plt.bar(feature_names, feature_importance)
plt.xticks(rotation=30)
plt.ylabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

# Evaluate accuracy using all features
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_all = cross_val_score(rfc, X, y, cv=cv, scoring="accuracy").mean()
print(f"CV Accuracy (all features): {acc_all:.5f}\n")

# Test accuracy with increasing top-k features
ranked_idx = np.argsort(feature_importance)[::-1]
for k in range(1, X.shape[1] + 1):
    topk_idx = ranked_idx[:k]
    acc_topk = cross_val_score(rfc, X[:, topk_idx], y, cv=cv, scoring="accuracy").mean()
    topk_names = [feature_names[i] for i in topk_idx]
    print(f"Top {k} feature(s): {topk_names} -> CV Accuracy: {acc_topk:.5f}")

print('\nThe difference between SFS and Random Forest feature rankings occurs because SFS optimize feature subsets specifically for k-NN accuracy, \nwhile Random Forest importance ranks features based on how well they split the data, focusing on reducing impurity. \nSince different models evaluate feature relevance differently, their rankings and resulting accuracies naturally differ.')

In [ ]:
# Q5
# PCA for Dimensionality Reduction
# Apply PCA on the Iris dataset. Reduce the data to 2D using the first two principal components
# and visualize the transformed data. Show the variance explained by the first few components.

iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained by 2 components:", pca.explained_variance_ratio_.sum())

for label, target_name in enumerate(target_names):
    plt.scatter(
        X_pca[y == label, 0],
        X_pca[y == label, 1],
        label=target_name
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA of Iris Dataset")
plt.legend()
plt.show()

print('The scatter plot shows that the first principal component strongly separates the Setosa class from the others, \nwhile the second principal component helps partially distinguish Versicolor and Virginica, \nindicating that most class variance is captured in the first two components.')

In [ ]:
# Q6
# Linear Discriminant Analysis (LDA)
# Perform LDA on the Iris dataset to reduce dimensionality of the features while maintaining class separation.
# Visualize the dataset in the reduced dimension and compare LDA and PCA in terms of class separability.

iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_scaled, y)

# PCA Plot
plt.subplot(1, 2, 1)
for label, target_name in enumerate(target_names):
    plt.scatter(
        X_pca[y == label, 0],
        X_pca[y == label, 1],
        label=target_name
    )
plt.title("PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(frameon=True, edgecolor="black")

# LDA Plot
plt.subplot(1, 2, 2)
for label, target_name in enumerate(target_names):
    plt.scatter(
        X_lda[y == label, 0],
        X_lda[y == label, 1],
        label=target_name
    )
plt.title("LDA")
plt.xlabel("LD1")
plt.ylabel("LD2")
plt.legend(frameon=True, edgecolor="black")

plt.tight_layout()
plt.show()

print('PCA captures directions of maximum variance but does not use class labels, resulting in some overlap between classes. \nLDA, being supervised, uses label information to maximize class separation, leading to clearer distinction between Iris species in the reduced space.')

In [ ]:
# Q7
# Feature Selection with Embedded Methods: Decision Trees
# Use a decision tree classifier to perfrom feature selection on the Iris dataset. Evaluate which
# features are selected by the model during tree construction. Visualize the tree to see selected features

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X, y)

# Features used in splits
used_feature_indices = np.unique(tree.tree_.feature[tree.tree_.feature != -2])
used_features = [feature_names[i] for i in used_feature_indices]
print("Features used in tree splits (selected features):")
print(used_features)

# Feature importances
print('\nFeature Importance:')
for name, importance in sorted(zip(feature_names, tree.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f'{name}: {importance:.5f}')

# Visualize tree
plot_tree(tree, feature_names=feature_names, class_names=target_names, filled=True)
plt.show()


In [ ]:
# Q8
# Correlation-Based Feature Selection (CFS)
# Implement CFS using the Iris dataset. Rank the features based on their correlation with the class label and with other features.

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df['class'] = y

corr_with_class_labels = df[feature_names].corrwith(df['class']).abs()
feature_corr = df[feature_names].corr().abs()
avg_corr_with_others = feature_corr.apply(lambda col: (col.sum()-1) / (len(feature_names)-1))
cfs_score = corr_with_class_labels - avg_corr_with_others

results = pd.DataFrame({
    'Feature': feature_names,
    'Correlation with class label': corr_with_class_labels.values,
    'Average correlation with other features': avg_corr_with_others.values,
    'CFS Score': cfs_score.values
}).sort_values(by='CFS Score', ascending=False)

print(results)


In [ ]:
# Q9
# Lasso Regularization for Feature Selection
# Use L1 Regularization with Logistic Regression on the Iris dataset to select features that are the most predictive of the class label.

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42)
model.fit(X_scaled, y)
coefficients = model.coef_

selected_features = set()
for i in range(coefficients.shape[0]):
    for j in range(coefficients.shape[1]):
        if coefficients[i][j] != 0:
            selected_features.add(feature_names[j])

print('Selected features:')
print(list(selected_features))